# Data Cleaning for AlertaRio Table

In [1]:
# !pip install pandas numpy

### Import Modules

In [2]:
import pandas as pd
import numpy as np

### Load Data

In [3]:
df = pd.read_csv('../../../../../data/meteorologia/raw/clima_pluviometro/taxa_precipitacao_alertario.csv')

display(df.head())

,primary_key,id_estacao,acumulado_chuva_15_min,acumulado_chuva_1_h,acumulado_chuva_4_h,acumulado_chuva_24_h,acumulado_chuva_96_h,horario,data_particao
0,1_2003-09-29 16:03:20,1,0.0,0.0,0.0,43.2,43.2,16:03:20,2003-09-29
1,1_2000-04-30 06:48:20,1,0.0,0.0,0.0,0.0,0.0,06:48:20,2000-04-30
2,1_2009-06-27 16:33:20,1,0.0,0.0,0.0,7.6,13.0,16:33:20,2009-06-27
3,1_2009-06-28 10:03:20,1,0.0,0.0,0.0,27.4,40.4,10:03:20,2009-06-28
4,1_2010-05-30 18:15:00,1,0.0,0.0,0.0,0.0,4.0,18:15:00,2010-05-30


### Drop columns

In [5]:
if 'Unnamed: 0' in df:
    df.drop('Unnamed: 0', axis=1, inplace=True)

### Extract fields from primary_key by splitting it

In [6]:
df[["id_estacao_extracted", "datetime_extracted"]] = df["primary_key"].str.split("_", n=1, expand=True)

### Convert data types

#### Convert extracted datetime into datetime format

In [7]:
df["datetime_extracted"] = pd.to_datetime(df["datetime_extracted"], format='ISO8601')

#### Convert datetime columns 

In [8]:
df["data_particao"] = pd.to_datetime(df["data_particao"])
df["horario"] = pd.to_timedelta(df["horario"])

#### Convert numeric columns

In [9]:
df["id_estacao_extracted"] = pd.to_numeric(df["id_estacao_extracted"])
df["id_estacao"] = pd.to_numeric(df["id_estacao"])

num_cols = [
    'acumulado_chuva_15_min',
    'acumulado_chuva_1_h',
    'acumulado_chuva_4_h',
    'acumulado_chuva_24_h',
    'acumulado_chuva_96_h'
]

df[num_cols] = df[num_cols].apply(pd.to_numeric)

### Handle missing values

#### Percentage of Missing Values per column

In [10]:
print('Percentual missing in each row:')
display(df.isna().mean().round(4).to_frame('Missing (%)') * 100)

Percentual missing in each row:


,Missing (%)
primary_key,0.00
id_estacao,0.00
acumulado_chuva_15_min,0.38
acumulado_chuva_1_h,0.39
acumulado_chuva_4_h,0.43
acumulado_chuva_24_h,0.49
acumulado_chuva_96_h,0.62
horario,0.02
data_particao,0.00
id_estacao_extracted,0.00


#### Drop rows missing `primary_key`

In [11]:
print(f'Dropping missing primary_key: {df["primary_key"].isna().sum()} rows ...')
df.dropna(subset=["primary_key"], inplace=True)  # Remove rows with missing primary_key

Dropping missing primary_key: 0 rows ...


#### Fill missing values in `horario` column with pd.to_timedelta(0.0)

In [12]:
df['horario'] = df['horario'].fillna(pd.to_timedelta(0.0))

#### Fill missing values in numeric columns with 0.0

In [13]:
num_cols = [
    'acumulado_chuva_15_min',
    'acumulado_chuva_1_h',
    'acumulado_chuva_4_h',
    'acumulado_chuva_24_h',
    'acumulado_chuva_96_h'
]

df[num_cols] = df[num_cols].fillna(0.0)  # Fill missing numerical values with 0

### Validate that extracted components match original columns

In [14]:
df["datetime_combined"] = df["data_particao"] + df["horario"].fillna(pd.to_timedelta(0))
df["is_consistent"] = (df["id_estacao"] == df["id_estacao_extracted"]) & (df["datetime_extracted"] == df["datetime_combined"])

if (~df['is_consistent']).sum():
    print('Inconsistent rows:\n')
    display(df[~df["is_consistent"]])
else:
    print('No inconsistent rows found.')

No inconsistent rows found.


### Drop inconsistent rows

In [15]:
if (~df['is_consistent']).sum():
    df = df[df["is_consistent"]]

### Drop Temporary columns

In [16]:
df = df.drop(columns=["id_estacao_extracted", "datetime_combined", "is_consistent"])

### Rename timestamp column

In [17]:
df.rename(columns={'datetime_extracted': 'timestamp'}, inplace=True)

### Remove duplicates

In [18]:
print(f'Duplicates of primary_key: {(df["primary_key"].value_counts() != 1).sum()}')

df.drop_duplicates(subset=["primary_key"], inplace=True)

Duplicates of primary_key: 33


### Sort data

In [19]:
df.sort_values('timestamp', inplace=True)

### Save cleaned data

In [20]:

df.to_csv("../../../../../data/meteorologia/clean/clima_pluviometro/taxa_precipitacao_alertario.csv", index=False)  # Optimized storage format

print("Data cleaning complete. Cleaned dataset saved.")

display(df.head())

Data cleaning complete. Cleaned dataset saved.


,primary_key,id_estacao,acumulado_chuva_15_min,acumulado_chuva_1_h,acumulado_chuva_4_h,acumulado_chuva_24_h,acumulado_chuva_96_h,horario,data_particao,timestamp
26778471,5_1997-01-01 01:00:40,5,0.0,0.0,0.0,0.0,0.0,0 days 01:00:40,1997-01-01,1997-01-01 01:00:40
15393572,13_1997-01-01 01:01:00,13,0.0,0.0,0.0,0.0,0.0,0 days 01:01:00,1997-01-01,1997-01-01 01:01:00
3914966,15_1997-01-01 01:01:20,15,0.0,0.0,0.0,0.0,0.0,0 days 01:01:20,1997-01-01,1997-01-01 01:01:20
12236785,7_1997-01-01 01:01:40,7,0.0,0.0,0.0,0.0,0.0,0 days 01:01:40,1997-01-01,1997-01-01 01:01:40
25102029,23_1997-01-01 01:02:00,23,0.0,0.0,0.0,0.0,0.0,0 days 01:02:00,1997-01-01,1997-01-01 01:02:00
